# Data Quality Assessment — Microsoft Fabric Lakehouse

**Author:** Jennifer Kelleman  
**Platform:** Microsoft Fabric (Spark Runtime 1.3+)  
**Purpose:** Scan Bronze Lakehouse tables and produce a data quality report by domain.

## What This Notebook Does

This notebook runs a structured quality assessment across all Bronze Delta tables in the attached Lakehouse. For each table, it measures:

- **Completeness** — what percentage of expected fields have non-null values
- **Uniqueness** — what percentage of records have a unique primary key
- **Timeliness** — how recent is the most recent record (vs. expected SLA)
- **Consistency** — do values conform to expected formats (dates, codes, ranges)
- **Volume** — are record counts within expected bounds

Results are written to `Tables/dq_assessment_results` and used by the Power BI Data Quality Dashboard.

In [ ]:
# ============================================================
# CELL 1: Imports and Configuration
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, TimestampType, BooleanType
from delta.tables import DeltaTable
import json
from datetime import datetime, timezone

# ── Configuration ──────────────────────────────────────────
# Update these values for your Lakehouse
LAKEHOUSE_NAME = "enterprise_lakehouse"  # Your Lakehouse display name
BRONZE_PREFIX = "bronze_"               # Prefix used to identify Bronze tables
RESULTS_TABLE = "dq_assessment_results" # Output table for quality results
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

# Domain metadata: maps table prefix → domain name
DOMAIN_MAP = {
    "bronze_customers":    "Customer",
    "bronze_orders":       "Finance",
    "bronze_transactions": "Finance",
    "bronze_products":     "Product",
    "bronze_employees":    "HR",
    "bronze_support":      "Operations",
    "bronze_events":       "Customer",
}

# Primary key definitions per table
PK_MAP = {
    "bronze_customers":    "customer_id",
    "bronze_orders":       "order_id",
    "bronze_transactions": "transaction_id",
    "bronze_products":     "product_id",
    "bronze_employees":    "employee_id",
    "bronze_support":      "case_id",
    "bronze_events":       "event_id",
}

# Expected freshness SLA: max hours since last record
FRESHNESS_SLA_HOURS = {
    "bronze_customers":    48,   # Updated twice daily
    "bronze_orders":       24,
    "bronze_transactions": 4,    # Near-real-time requirement
    "bronze_products":     168,  # Weekly catalog update
    "bronze_employees":    168,
    "bronze_support":      4,
    "bronze_events":       2,    # High-frequency event stream
}

print(f"Run ID: {RUN_ID}")
print(f"Assessing tables with prefix: {BRONZE_PREFIX}")

In [ ]:
# ============================================================
# CELL 2: Discover Bronze Tables
# ============================================================

# List all Delta tables in the Lakehouse
all_tables = [t.name for t in spark.catalog.listTables() 
              if t.name.startswith(BRONZE_PREFIX)]

print(f"Discovered {len(all_tables)} Bronze tables:")
for t in sorted(all_tables):
    print(f"  • {t}")

In [ ]:
# ============================================================
# CELL 3: Core Quality Assessment Function
# ============================================================

def assess_table_quality(table_name: str, run_id: str) -> dict:
    """
    Runs a comprehensive quality assessment on a single Delta table.
    
    Returns a dict of quality metrics suitable for writing to the results table.
    """
    result = {
        "run_id":           run_id,
        "assessment_ts":    datetime.now(timezone.utc),
        "table_name":       table_name,
        "domain":           DOMAIN_MAP.get(table_name, "Unknown"),
        "primary_key":      PK_MAP.get(table_name),
        "freshness_sla_h":  FRESHNESS_SLA_HOURS.get(table_name, 24),
    }
    
    try:
        df = spark.table(table_name)
        total = df.count()
        result["total_records"] = total
        
        if total == 0:
            result.update({
                "completeness_score": 0.0,
                "uniqueness_score":   0.0,
                "freshness_score":    0.0,
                "overall_score":      0.0,
                "is_empty":           True,
                "quality_tier":       "CRITICAL",
                "assessment_notes":   "Table is empty."
            })
            return result
        
        result["is_empty"] = False
        columns = df.columns
        non_audit_cols = [c for c in columns if not c.startswith("_")]
        
        # ── COMPLETENESS ──────────────────────────────────────
        # % of non-null values across all non-audit columns
        null_counts = df.select([
            F.sum(F.col(c).isNull().cast("int")).alias(c) 
            for c in non_audit_cols
        ]).collect()[0].asDict()
        
        total_cells = total * len(non_audit_cols)
        total_nulls = sum(null_counts.values())
        completeness = 1.0 - (total_nulls / total_cells) if total_cells > 0 else 0.0
        result["completeness_score"] = round(completeness, 4)
        result["null_cell_count"]    = int(total_nulls)
        result["columns_assessed"]   = len(non_audit_cols)
        
        # ── UNIQUENESS ────────────────────────────────────────
        # % unique primary key values
        pk = result["primary_key"]
        if pk and pk in columns:
            distinct_pks = df.filter(F.col(pk).isNotNull()).select(pk).distinct().count()
            non_null_pks = df.filter(F.col(pk).isNotNull()).count()
            uniqueness = distinct_pks / non_null_pks if non_null_pks > 0 else 0.0
            result["uniqueness_score"]    = round(uniqueness, 4)
            result["duplicate_pk_count"]  = int(non_null_pks - distinct_pks)
        else:
            result["uniqueness_score"]    = None  # No PK defined or PK not in table
            result["duplicate_pk_count"]  = None
        
        # ── FRESHNESS ─────────────────────────────────────────
        # Hours since most recent _ingest_timestamp
        if "_ingest_timestamp" in columns:
            max_ts_row = df.agg(F.max("_ingest_timestamp").alias("max_ts")).collect()[0]
            max_ts = max_ts_row["max_ts"]
            if max_ts:
                now = datetime.now(timezone.utc)
                hours_stale = (now - max_ts.replace(tzinfo=timezone.utc)).total_seconds() / 3600
                sla_hours = result["freshness_sla_h"]
                freshness = max(0.0, 1.0 - (hours_stale / sla_hours))
                result["freshness_score"]  = round(freshness, 4)
                result["hours_stale"]      = round(hours_stale, 1)
                result["latest_record_ts"] = max_ts.isoformat()
            else:
                result["freshness_score"]  = 0.0
                result["hours_stale"]      = None
                result["latest_record_ts"] = None
        else:
            result["freshness_score"]  = None  # No ingest timestamp
            result["hours_stale"]      = None
            result["latest_record_ts"] = None
        
        # ── OVERALL SCORE ─────────────────────────────────────
        # Weighted average of available dimensions
        scores = [s for s in [
            result.get("completeness_score"),
            result.get("uniqueness_score"),
            result.get("freshness_score")
        ] if s is not None]
        
        overall = sum(scores) / len(scores) if scores else 0.0
        result["overall_score"] = round(overall, 4)
        
        # ── QUALITY TIER ──────────────────────────────────────
        if overall >= 0.90:
            tier = "EXCELLENT"
        elif overall >= 0.80:
            tier = "GOOD"
        elif overall >= 0.65:
            tier = "ACCEPTABLE"
        elif overall >= 0.50:
            tier = "POOR"
        else:
            tier = "CRITICAL"
        
        result["quality_tier"]   = tier
        result["assessment_notes"] = ""
        
        # Flag specific issues
        notes = []
        if completeness < 0.80:
            notes.append(f"Low completeness ({completeness:.1%}): review null fields in source.")
        if result.get("uniqueness_score", 1.0) is not None and result.get("uniqueness_score", 1.0) < 0.99:
            notes.append(f"Duplicate PKs detected ({result['duplicate_pk_count']} dupes): investigate source deduplication.")
        if result.get("hours_stale") is not None and result["hours_stale"] > result["freshness_sla_h"]:
            notes.append(f"Stale data ({result['hours_stale']:.1f}h > {result['freshness_sla_h']}h SLA): check pipeline schedule.")
        
        result["assessment_notes"] = " | ".join(notes) if notes else "All checks passed."
        
    except Exception as e:
        result.update({
            "total_records":     None,
            "completeness_score": 0.0,
            "uniqueness_score":   0.0,
            "freshness_score":    0.0,
            "overall_score":      0.0,
            "quality_tier":       "ERROR",
            "assessment_notes":   f"Assessment failed: {str(e)}",
            "is_empty":           None
        })
    
    return result

print("Quality assessment function defined.")

In [ ]:
# ============================================================
# CELL 4: Run Assessment Across All Bronze Tables
# ============================================================

results = []

for table_name in sorted(all_tables):
    print(f"Assessing {table_name}...", end=" ")
    result = assess_table_quality(table_name, RUN_ID)
    results.append(result)
    print(f"{result['quality_tier']} (score: {result['overall_score']:.1%})")

print(f"\nAssessment complete. {len(results)} tables evaluated.")

In [ ]:
# ============================================================
# CELL 5: Write Results to Delta Table
# ============================================================

from pyspark.sql.types import (
    StructType, StructField, StringType, FloatType, 
    IntegerType, TimestampType, BooleanType
)

schema = StructType([
    StructField("run_id",              StringType(),    False),
    StructField("assessment_ts",       TimestampType(), False),
    StructField("table_name",          StringType(),    False),
    StructField("domain",              StringType(),    True),
    StructField("primary_key",         StringType(),    True),
    StructField("freshness_sla_h",     IntegerType(),   True),
    StructField("total_records",       IntegerType(),   True),
    StructField("is_empty",            BooleanType(),   True),
    StructField("columns_assessed",    IntegerType(),   True),
    StructField("null_cell_count",     IntegerType(),   True),
    StructField("duplicate_pk_count",  IntegerType(),   True),
    StructField("hours_stale",         FloatType(),     True),
    StructField("latest_record_ts",    StringType(),    True),
    StructField("completeness_score",  FloatType(),     True),
    StructField("uniqueness_score",    FloatType(),     True),
    StructField("freshness_score",     FloatType(),     True),
    StructField("overall_score",       FloatType(),     False),
    StructField("quality_tier",        StringType(),    False),
    StructField("assessment_notes",    StringType(),    True),
])

results_df = spark.createDataFrame(results, schema=schema)

results_df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(RESULTS_TABLE)

print(f"Results written to Tables/{RESULTS_TABLE}")

In [ ]:
# ============================================================
# CELL 6: Summary Report (Executive View)
# ============================================================

print("=" * 65)
print(f"  DATA QUALITY ASSESSMENT — RUN {RUN_ID}")
print("=" * 65)

results_df_sorted = results_df.orderBy(F.col("overall_score").asc())

# Overall estate summary
avg_score = results_df.agg(F.avg("overall_score")).collect()[0][0]
tier_counts = results_df.groupBy("quality_tier").count().collect()

print(f"\nEstate Average Quality Score: {avg_score:.1%}")
print("\nDistribution by Quality Tier:")
for row in sorted(tier_counts, key=lambda r: ["CRITICAL","ERROR","POOR","ACCEPTABLE","GOOD","EXCELLENT"].index(r["quality_tier"])):
    bar = "█" * row["count"]
    print(f"  {row['quality_tier']:12s} {bar} ({row['count']})")

print("\nTables Requiring Attention:")
critical_tables = results_df.filter(
    F.col("quality_tier").isin(["CRITICAL", "POOR", "ERROR"])
).select("table_name", "overall_score", "quality_tier", "assessment_notes").collect()

if critical_tables:
    for row in critical_tables:
        print(f"  ⚠  {row['table_name']}: {row['quality_tier']} ({row['overall_score']:.1%})")
        print(f"     → {row['assessment_notes']}")
else:
    print("  ✓  No tables in CRITICAL or POOR tier.")

print("\nDomain Summary:")
domain_summary = results_df.groupBy("domain").agg(
    F.avg("overall_score").alias("avg_score"),
    F.count("table_name").alias("table_count")
).orderBy(F.col("avg_score").asc()).collect()

for row in domain_summary:
    score_bar = "█" * int(row["avg_score"] * 20)
    print(f"  {row['domain']:14s} {score_bar:20s} {row['avg_score']:.1%} ({row['table_count']} tables)")

print("=" * 65)